# Jev System One Models

This notebook demonstrates production-grade use-cases using **Jev**, a specialized model family optimized for high-performance, deterministic classification, risk scoring, and routing task queries.

### What is Jev?
Jev is a state-of-the-art **System One** model developed by **Typesafe**. Unlike typical LLMs that engage in verbose, slow, and expensive auto-regressive decoding (System Two thinking), Jev is built specifically for instantaneous, structured, and highly calibrated categorical predictions. Jev models map input states directly to exact distributions over discrete primitives (such as binary flags, multi-class selections, and numerical score ranges).

---


In this notebook, we explore 12 real-world enterprise applications leveraging Jev's instant routing and guardrail capabilities, both through direct HTTP integrations and via the open-source **Laya** routing agent library.

In [27]:
from google.colab import userdata
BEATAPI_API_KEY=userdata.get('TYPESAFE_API_KEY')

In [28]:
import requests
import json

# The API key is already loaded as TYPESAFE_API_KEY

url = "https://api.beatapi.io/v1/systemone"
headers = {
    "Authorization": f"Bearer {BEATAPI_API_KEY}", # Using BEATAPI_API_KEY from previous cell
    "Content-Type": "application/json"
}

json_payload = {
    "model": "jev-1.13-free",
    "state": "Task: clean up inactive accounts before the quarterly report.\nProposed tool call: delete_rows(table=\"customers\", where=\"last_login < 2023-01-01\")\nContext: the customers table has 48,210 rows and no backup was taken today.",
    "questions": {
      "safe_to_run": {
        "type": "noul",
        "instructions": "Is this action safe to run without a human approving it first?",
        "criteria": {
          "true": "Reversible or low-impact, and clearly within the stated task.",
          "false": "Destructive, irreversible, or broader than the task requires."
        }
      }
    }
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(json_payload))
    response.raise_for_status() # Raise an exception for HTTP errors
    response_data = response.json()

    # Corrected key from 'nouls' to 'answers'
    safe_to_run = response_data['answers']['safe_to_run']['noul']
    print(f"Is it safe to run without human approval: {safe_to_run}")

except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")
    if e.response is not None:
        print(f"Response content: {e.response.text}")
except KeyError:
    print("Error parsing API response. 'answers' or 'safe_to_run' key not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Is it safe to run without human approval: 0.05


In [29]:
response_data

{'answers': {'safe_to_run': {'noul': 0.05, 'type': 'noul'}},
 'id': 'task_0yib6ae8L2WjhelvAyPv8YmmzqHE0H0g',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 384, 'output_tokens': 22}}

## Section 1: Jev model implementation using BeatAPI Raw HTTP Implementations

In this section, we showcase 10 different production use-cases using raw HTTP requests via the `requests` library to query `https://api.beatapi.io/v1/systemone`.

In [31]:
import requests
import json

def query_beatapi(state, questions, model="jev-1.13-free"):
    """
    Helper function to send structured queries to the BeatAPI systemone endpoint.
    """
    url = "https://api.beatapi.io/v1/systemone"
    headers = {
        "Authorization": f"Bearer {BEATAPI_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "state": state,
        "questions": questions
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        if e.response is not None:
            print(f"Response content: {e.response.text}")
        return None

# Quick test using Customer Support Ticket Triage state
customer_email = (
    "Hey support, I ordered a premium leather jacket (Order #8932) last Monday. "
    "It was supposed to arrive yesterday. The tracking link says it's still stuck "
    "in the warehouse. I need this by Friday for a wedding. Please help."
)

questions = {
    "is_urgent": {
        "type": "noul",
        "instructions": "Does this require immediate, escalated customer support response?",
        "criteria": {
            "true": "Requires immediate attention and escalations.",
            "false": "Standard support behavior."
        }
    },
    "department_routing": {
        "type": "choice",
        "choices": ["Billing", "Shipping & Logistics", "Product Returns", "Technical Support"],
        "instructions": "Which internal department should handle this customer issue?",
        "criteria": {
            "Billing": "Handles payment issues, billing cycles, or transaction disputes.",
            "Shipping & Logistics": "Deals with delivery issues, package tracking, and delays.",
            "Product Returns": "Assists with exchanging goods and returning merchandise.",
            "Technical Support": "Handles technical issues, software configuration, and account access."
        }
    },
    "frustration_level": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the customer's frustration level from 1 to 5",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

response_data = query_beatapi(customer_email, questions)
if response_data:
    answers = response_data.get('answers', {})
    print(f"Is Urgent (Noul): {answers.get('is_urgent', {}).get('noul')}")
    print(f"Department Routing: {answers.get('department_routing', {}).get('choice')}")
    print(f"Frustration Level: {answers.get('frustration_level', {}).get('score')}")

Is Urgent (Noul): 0.61
Department Routing: Shipping & Logistics
Frustration Level: 2.05


In [32]:
response_data

{'answers': {'department_routing': {'choice': 'Shipping & Logistics',
   'confidence': 1,
   'probabilities': {'Billing': 0,
    'Product Returns': 0,
    'Shipping & Logistics': 1,
    'Technical Support': 0},
   'type': 'choice'},
  'frustration_level': {'confidence': 0.6,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0.21, '2': 0.53, '3': 0.26, '4': 0},
   'score': 2.05,
   'type': 'score'},
  'is_urgent': {'noul': 0.61, 'type': 'noul'}},
 'id': 'task_JTYJzWrU81PdJrGzX6dIdUmXdLvaheJH',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 529, 'output_tokens': 90}}

In [33]:
import requests
import json
from typing import Dict, Any, Optional

def query_jev_model(
    state: str,
    questions: Dict[str, Any],
    model: str = "jev-1.13-free",
    api_key: Optional[str] = None
) -> Optional[Dict[str, Any]]:
    """
    Queries the Typesafe Jev Model System One endpoint with modular structured state and questions.

    This function serves as the central, production-grade interface for dispatching requests
    to the System One reasoning API. It guarantees standard HTTP header configuration, enforces
    robust connection/timeout boundaries, handles structural API changes gracefully, and contains
    diagnostic trace outputs for runtime failures.

    Args:
        state (str): The primary input context, body text, or serialized metrics to analyze.
        questions (Dict[str, Any]): A structured dictionary defining binary (noul), categorical (choice),
                                    or numerical range (score) questions alongside evaluation criteria.
        model (str): The unique identifier of the target Jev model (defaults to 'jev-1.13-free').
        api_key (Optional[str]): Secure API authorization bearer token. If not provided, fallback
                                 resolves automatically to the global BEATAPI_API_KEY environment variable.

    Returns:
        Optional[Dict[str, Any]]: A dictionary containing the raw predictions inside the 'answers' key
                                  if successful, or None if the request fails.
    """
    # 1. Resolve Auth credentials using priority routing (argument API key -> global state key)
    target_api_key = api_key or BEATAPI_API_KEY
    if not target_api_key:
        print("[Error] Jev Request Failed: No API key was provided or found in the environment.")
        return None

    # 2. Establish connection boundaries and endpoint URLs
    endpoint_url = "https://api.beatapi.io/v1/systemone"
    request_headers = {
        "Authorization": f"Bearer {target_api_key}",
        "Content-Type": "application/json"
    }

    # 3. Assemble the deterministic payload structure required by Jev
    request_payload = {
        "model": model,
        "state": state,
        "questions": questions
    }

    # 4. Dispatch the remote execution payload with network error isolation
    try:
        response = requests.post(endpoint_url, headers=request_headers, json=request_payload)
        response.raise_for_status() # Raise an exception for HTTP errors (e.g. 400, 401, 500)
        return response.json()
    except requests.exceptions.RequestException as error:
        print(f"[Error] API Network request failed: {error}")
        if error.response is not None:
            print(f"[Diagnostics] Server Response Body: {error.response.text}")
        return None
    except Exception as unexpected_error:
        print(f"[Error] Unexpected processing exception: {unexpected_error}")
        return None

### Use Case 1: Customer Support Ticket Triage

This use case evaluates inbound customer service messages. It automatically checks if the request is an emergency requiring immediate escalation, routes the ticket to the correct department (Billing, Technical Support, or Shipping), and assigns an overall anger score to prioritize high-frustration situations.

In [35]:
# --- Use Case 1: Customer Support Ticket Triage ---
# This use case evaluates incoming customer support messages to prioritize urgent issues,
# route them to the specialized internal department, and evaluate customer sentiment.

customer_ticket_state = (
    "My internet has been down for 3 days! I can't work from home. This is an emergency! "
    "I need someone to fix this NOW. "
)

# Define the deterministic questions and categorical criteria mapping for the Jev model
support_triage_questions = {
    "is_emergency": {
        "type": "noul",
        "instructions": "Is the customer issue an active emergency?",
        "criteria": {
            "true": "The issue requires immediate, critical attention due to severe impact or danger.",
            "false": "The issue is standard and does not pose an immediate threat or severe disruption."
        }
    },
    "department_routing": {
        "type": "choice",
        "choices": ["Billing", "Technical Support", "Shipping"],
        "instructions": "Which department should handle this ticket?",
        "criteria": {
            "Billing": "Deals with invoices, payments, and account charges.",
            "Technical Support": "Handles product malfunctions, software issues, or technical guidance.",
            "Shipping": "Manages delivery, tracking, and logistics."
        }
    },
    "anger_level": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the customer's anger level from 1 (calm) to 5 (enraged)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

# Query the System One endpoint using our modular and safe helper function
print("[INFO] Dispatching state payload to Jev Model...")
response = query_jev_model(state=customer_ticket_state, questions=support_triage_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely from the structured response payload
    is_emergency = answers.get('is_emergency', {}).get('noul')
    department_routing = answers.get('department_routing', {}).get('choice')
    anger_level = answers.get('anger_level', {}).get('score')

    # Print verified output
    print(f"-> Prediction Success:")
    print(f"   Is Emergency Probability: {is_emergency}")
    print(f"   Routed Department:       {department_routing}")
    print(f"   Anger Score (1-5):       {anger_level}")
else:
    print("[Error] Failed to get valid response from the Jev API.")

[INFO] Dispatching state payload to Jev Model...
-> Prediction Success:
   Is Emergency Probability: 0.33
   Routed Department:       Technical Support
   Anger Score (1-5):       3.05


In [36]:
response

{'answers': {'anger_level': {'confidence': 0.84,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0, '2': 0.07, '3': 0.8, '4': 0.13},
   'score': 3.05,
   'type': 'score'},
  'department_routing': {'choice': 'Technical Support',
   'confidence': 1,
   'probabilities': {'Billing': 0, 'Shipping': 0, 'Technical Support': 1},
   'type': 'choice'},
  'is_emergency': {'noul': 0.33, 'type': 'noul'}},
 'id': 'task_KIr33NEMnWuJY7U8wCQkPpY6raXseDB7',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 506, 'output_tokens': 77}}

### Use Case 2: Content Moderation and Brand Safety

This use case scans user-generated social posts or comments to detect explicit hate speech, classify the specific community guidelines violation (spam, harassment, profanity, or hate speech), and compute a toxicity rating from 1 to 10.

In [37]:
# --- Use Case 2: Content Moderation and Brand Safety ---
# This use case analyzes user-generated posts to identify hate speech,
# categorize community violations, and assign a calibrated toxicity score.

user_comment_state = "This website is terrible and anyone who uses it is an idiot. You should all go to hell!"

# Structured validation schema and categorical guidelines
content_moderation_questions = {
    "flag_hate_speech": {
        "type": "noul",
        "instructions": "Does this comment contain explicit hate speech?",
        "criteria": {
            "true": "The comment contains derogatory, discriminatory, or hateful language targeting a specific group or individual.",
            "false": "The comment does not contain explicit hate speech."
        }
    },
    "violation_type": {
        "type": "choice",
        "choices": ["spam", "harassment", "profanity", "hate_speech"],
        "instructions": "Categorize the primary violation type.",
        "criteria": {
            "spam": "Unsolicited or repetitive content.",
            "harassment": "Targeted abusive behavior.",
            "profanity": "Offensive language.",
            "hate_speech": "Expressions of hatred, discrimination, or violence against a protected group."
        }
    },
    "toxicity_level": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Rate the toxicity level from 1 (mild) to 10 (extreme)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

# Query the Jev Model via our unified request interface
print("[INFO] Dispatching content state to Jev Model...")
response = query_jev_model(state=user_comment_state, questions=content_moderation_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Extract outputs securely
    flag_hate_speech = answers.get('flag_hate_speech', {}).get('noul')
    violation_type = answers.get('violation_type', {}).get('choice')
    toxicity_level = answers.get('toxicity_level', {}).get('score')

    # Display parsing diagnostics
    print(f"-> Moderation Flagged:")
    print(f"   Hate Speech Likelihood: {flag_hate_speech}")
    print(f"   Violation Category:     {violation_type}")
    print(f"   Calibrated Toxicity:    {toxicity_level}")
else:
    print("[Error] Failed to process content moderation checks.")

[INFO] Dispatching content state to Jev Model...
-> Moderation Flagged:
   Hate Speech Likelihood: 0.29
   Violation Category:     harassment
   Calibrated Toxicity:    7.32


In [38]:
response

{'answers': {'flag_hate_speech': {'noul': 0.29, 'type': 'noul'},
  'toxicity_level': {'confidence': 0.62,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0,
    '1': 0,
    '2': 0.01,
    '3': 0.01,
    '4': 0.01,
    '5': 0.04,
    '6': 0.13,
    '7': 0.3,
    '8': 0.37,
    '9': 0.13},
   'score': 7.32,
   'type': 'score'},
  'violation_type': {'choice': 'harassment',
   'confidence': 0.38,
   'probabilities': {'harassment': 0.54,
    'hate_speech': 0.07,
    'profanity': 0.39,
    'spam': 0},
   'type': 'choice'}},
 'id': 'task_CNr7bpcVeBy1XWkyAGhFwJXk69Gnsxpg',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 539, 'output_tokens': 92}}

### Use Case 3: Resume Screening and Candidate Pre-Qualification

This use case automates the screening of job applications by checking if candidates meet strict experience minimums, categorizing their professional seniority tier (Junior, Mid, Senior, or Lead), and scoring their skill set alignment against a target job profile.

In [39]:
# --- Use Case 3: Resume Screening and Candidate Pre-Qualification ---
# Parses unformatted candidate experience metadata to verify strict requirements,
# classify seniority tiers, and rate alignment with specialized roles.

resume_text_state = (
    "John Doe. 10 years experience in Machine Learning. "
    "Led 3 major AI projects. PhD in Computer Science. "
    "Skills: Python, TensorFlow, PyTorch, AWS, Azure."
)

# High-context extraction structure
resume_screening_questions = {
    "mandatory_experience_met": {
        "type": "noul",
        "instructions": "Does the applicant have at least 5 years of experience in Machine Learning?",
        "criteria": {
            "true": "Applicant explicitly states or implies 5 or more years of relevant experience.",
            "false": "Applicant states less than 5 years or does not provide enough information."
        }
    },
    "seniority_tier": {
        "type": "choice",
        "choices": ["Junior", "Mid", "Senior", "Lead"],
        "instructions": "Categorize the candidate's seniority tier.",
        "criteria": {
            "Junior": "Less than 3 years of experience.",
            "Mid": "3-5 years of experience.",
            "Senior": "5+ years of experience with leadership potential.",
            "Lead": "Extensive experience, proven leadership, and project management skills."
        }
    },
    "skill_alignment": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Rate the overall skill alignment with a typical Senior Machine Learning Engineer role (1-low, 10-high)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

print("[INFO] Screening incoming resume data...")
response = query_jev_model(state=resume_text_state, questions=resume_screening_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions
    experience_ok = answers.get('mandatory_experience_met', {}).get('noul')
    seniority = answers.get('seniority_tier', {}).get('choice')
    alignment_score = answers.get('skill_alignment', {}).get('score')

    print(f"-> Candidate Profile Analysis:")
    print(f"   Mandatory Target Met:  {experience_ok}")
    print(f"   Assessed Seniority:    {seniority}")
    print(f"   Objective Alignment:   {alignment_score}")
else:
    print("[Error] Candidate screening failed.")

[INFO] Screening incoming resume data...
-> Candidate Profile Analysis:
   Mandatory Target Met:  0.99
   Assessed Seniority:    Lead
   Objective Alignment:   7.52


In [40]:
response

{'answers': {'mandatory_experience_met': {'noul': 0.99, 'type': 'noul'},
  'seniority_tier': {'choice': 'Lead',
   'confidence': 0.88,
   'probabilities': {'Junior': 0, 'Lead': 0.91, 'Mid': 0, 'Senior': 0.09},
   'type': 'choice'},
  'skill_alignment': {'confidence': 0.75,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0,
    '1': 0,
    '2': 0,
    '3': 0,
    '4': 0,
    '5': 0.01,
    '6': 0.05,
    '7': 0.39,
    '8': 0.48,
    '9': 0.07},
   'score': 7.52,
   'type': 'score'}},
 'id': 'task_GjvSpgxGHAKD3yw2vgx5nOEOlz3UcdBl',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 575, 'output_tokens': 85}}

### Use Case 4: E-Commerce Fake Review Detection

This use case protects platform integrity by flagging incentivized or suspicious reviews (such as those mentioning receiving free items), identifying the main focus of the review (quality, price, shipping, or support), and scoring the review's estimated authenticity from 1 to 5.

In [42]:
# --- Use Case 4: E-Commerce Fake Review Detection ---
# Analyzes reviews to flag transactional or incentivized speech,
# classify topic contexts, and assess baseline reviews' integrity scores.

product_review_state = (
    "This product is absolutely amazing! I got it for free "
    "as part of a promotion. Best purchase ever!"
)

fake_review_questions = {
    "flag_fraudulent_review": {
        "type": "noul",
        "instructions": "Is this review potentially fraudulent or incentivized?",
        "criteria": {
            "true": "The review mentions receiving the product for free, in exchange for a review, or shows other signs of unnatural positivity or repetition.",
            "false": "The review appears genuine and organic."
        }
    },
    "review_focus": {
        "type": "choice",
        "choices": ["quality", "price", "shipping", "customer_service"],
        "instructions": "Classify the primary focus of the review.",
        "criteria": {
            "quality": "Focuses on the product's durability, performance, or materials.",
            "price": "Comments on the cost, value for money, or deals.",
            "shipping": "Mentions delivery speed, packaging, or logistics.",
            "customer_service": "Refers to interactions with support or sales."
        }
    },
    "authenticity_rating": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Estimate the authenticity rating from 1 (very fake) to 5 (very authentic)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Evaluating review integrity...")
response = query_jev_model(state=product_review_state, questions=fake_review_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    flag_fraud = answers.get('flag_fraudulent_review', {}).get('noul')
    review_focus = answers.get('review_focus', {}).get('choice')
    authenticity = answers.get('authenticity_rating', {}).get('score')

    print(f"-> Anti-Fraud Assessment:")
    print(f"   Incentivized Flag Likelihood: {flag_fraud}")
    print(f"   Detected Focus Topic:        {review_focus}")
    print(f"   Authenticity Confidence:     {authenticity}")
else:
    print("[Error] Review classification failed.")

[INFO] Evaluating review integrity...
-> Anti-Fraud Assessment:
   Incentivized Flag Likelihood: 0.91
   Detected Focus Topic:        price
   Authenticity Confidence:     0.61


In [43]:
response

{'answers': {'authenticity_rating': {'confidence': 0.49,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.51, '1': 0.4, '2': 0.07, '3': 0.02, '4': 0},
   'score': 0.61,
   'type': 'score'},
  'flag_fraudulent_review': {'noul': 0.91, 'type': 'noul'},
  'review_focus': {'choice': 'price',
   'confidence': 0.92,
   'probabilities': {'customer_service': 0,
    'price': 0.94,
    'quality': 0.06,
    'shipping': 0},
   'type': 'choice'}},
 'id': 'task_cxPPrdtApnSI78AlxA3Eh2oQqRKhy7rf',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 525, 'output_tokens': 83}}

### Use Case 5: Sentiment Analysis of Social Media Posts

This use case tracks brand perception across social networks. It determines if a post expresses negative sentiment, categorizes the tone as positive, negative, or neutral, and rates the overall emotional intensity of the expression.

In [44]:
# --- Use Case 5: Sentiment Analysis of Social Media Posts ---
# This step measures categorical sentiment, confirms polarity negativity, and rates overall intensity.

social_media_post_state = (
    "Just had the worst experience with my new phone. The battery dies so fast, "
    "and the camera is terrible. What a waste of money! #disappointed #techfail"
)

sentiment_analysis_questions = {
    "is_negative_sentiment": {
        "type": "noul",
        "instructions": "Does the social media post express negative sentiment?",
        "criteria": {
            "true": "The post contains words, phrases, or overall tone indicating dissatisfaction, anger, or sadness.",
            "false": "The post is neutral or positive in sentiment."
        }
    },
    "sentiment_category": {
        "type": "choice",
        "choices": ["positive", "negative", "neutral"],
        "instructions": "Categorize the overall sentiment of the post.",
        "criteria": {
            "positive": "Expresses approval, happiness, or satisfaction.",
            "negative": "Expresses disapproval, sadness, or frustration.",
            "neutral": "No strong positive or negative emotion is conveyed."
        }
    },
    "sentiment_intensity": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the intensity of the sentiment from 1 (mild) to 5 (extreme)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Dispatched state context to Jev Sentiment analysis...")
response = query_jev_model(state=social_media_post_state, questions=sentiment_analysis_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    is_negative = answers.get('is_negative_sentiment', {}).get('noul')
    sentiment_category = answers.get('sentiment_category', {}).get('choice')
    intensity = answers.get('sentiment_intensity', {}).get('score')

    print(f"-> Social Sentiment Report:")
    print(f"   Negative Polarity Value: {is_negative}")
    print(f"   Resolved Category:       {sentiment_category}")
    print(f"   Resolved Intensity:      {intensity}")
else:
    print("[Error] Failed to resolve sentiment scores.")

[INFO] Dispatched state context to Jev Sentiment analysis...
-> Social Sentiment Report:
   Negative Polarity Value: 0.99
   Resolved Category:       negative
   Resolved Intensity:      2.85


In [45]:
response

{'answers': {'is_negative_sentiment': {'noul': 0.99, 'type': 'noul'},
  'sentiment_category': {'choice': 'negative',
   'confidence': 1,
   'probabilities': {'negative': 1, 'neutral': 0, 'positive': 0},
   'type': 'choice'},
  'sentiment_intensity': {'confidence': 0.82,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0.02, '2': 0.14, '3': 0.81, '4': 0.03},
   'score': 2.85,
   'type': 'score'}},
 'id': 'task_0HuVL3v7hGcoQUL2wjYd7wEwT0QdGTxC',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 510, 'output_tokens': 78}}

### Use Case 6: Sales Lead Qualification and Intent Scoring

This use case analyzes inbound inquiry forms or chats to verify if the prospect has an immediate budget, identify where they are in their buying journey (researching, comparing, or ready to buy), and output an overall conversion score.

In [46]:
# --- Use Case 6: Sales Lead Qualification and Intent Scoring ---
# This use case evaluates inbound lead submissions to score immediate budget readiness,
# identify journey stages, and predict conversion probability.

sales_lead_state = (
    "Interested in your enterprise solution. Our budget is $100k, "
    "and we're looking to implement within the next 3 months. "
    "Currently evaluating vendors."
)

lead_qualification_questions = {
    "has_immediate_budget": {
        "type": "noul",
        "instructions": "Does the lead have an immediate budget for the solution?",
        "criteria": {
            "true": "The lead explicitly states a budget or indicates readiness for financial commitment.",
            "false": "The lead does not mention a budget or indicates they are still early in the financial planning phase."
        }
    },
    "buyer_journey_stage": {
        "type": "choice",
        "choices": ["researching", "comparing", "ready_to_buy"],
        "instructions": "Classify the lead's buyer journey stage.",
        "criteria": {
            "researching": "Gathering information and understanding options.",
            "comparing": "Evaluating different solutions and vendors.",
            "ready_to_buy": "Prepared to make a purchase decision and looking for final quotes."
        }
    },
    "conversion_potential": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Grade the lead's conversion potential from 1 (low) to 10 (high)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

print("[INFO] Qualifying sales lead with Jev...")
response = query_jev_model(state=sales_lead_state, questions=lead_qualification_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    has_budget = answers.get('has_immediate_budget', {}).get('noul')
    journey_stage = answers.get('buyer_journey_stage', {}).get('choice')
    conversion_score = answers.get('conversion_potential', {}).get('score')

    print(f"-> Lead Assessment:")
    print(f"   Immediate Budget Likelihood: {has_budget}")
    print(f"   Buyer Journey Stage:         {journey_stage}")
    print(f"   Conversion Potential:        {conversion_score}")
else:
    print("[Error] Lead qualification request failed.")

[INFO] Qualifying sales lead with Jev...
-> Lead Assessment:
   Immediate Budget Likelihood: 0.91
   Buyer Journey Stage:         comparing
   Conversion Potential:        6.2


In [47]:
response

{'answers': {'buyer_journey_stage': {'choice': 'comparing',
   'confidence': 0.99,
   'probabilities': {'comparing': 0.99,
    'ready_to_buy': 0.01,
    'researching': 0},
   'type': 'choice'},
  'conversion_potential': {'confidence': 0.73,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0,
    '1': 0,
    '2': 0,
    '3': 0.01,
    '4': 0.03,
    '5': 0.13,
    '6': 0.45,
    '7': 0.33,
    '8': 0.05,
    '9': 0},
   'score': 6.2,
   'type': 'score'},
  'has_immediate_budget': {'noul': 0.91, 'type': 'noul'}},
 'id': 'task_aWBwuW2e2DYbMPpWw0gKrvYC0h4nj5gX',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 545, 'output_tokens': 88}}

### Use Case 7: News and Article Bias Guardrails

This use case provides editorial protection by flagging unsubstantiated or misleading claims, detecting political or commercial biases (liberal, conservative, commercial promotion, etc.), and scoring the objective factual quality of the text.

In [48]:
# --- Use Case 7: News and Article Bias Guardrails ---
# Analyzes text copy or shared articles to flag potential media bias,
# detect loaded/misleading allegations, and rate overall factual quality.

news_article_state = (
    "Breaking news: The government's new policy, widely condemned by experts, "
    "is clearly an attempt to suppress dissent and control information. "
    "This authoritarian move will undoubtedly lead to widespread protests."
)

bias_guardrails_questions = {
    "has_misleading_claims": {
        "type": "noul",
        "instructions": "Does the text contain misleading claims or unsubstantiated accusations?",
        "criteria": {
            "true": "The text presents opinions as facts, uses loaded language, or makes claims without evidence.",
            "false": "The text is factual, balanced, and avoids deceptive language."
        }
    },
    "political_commercial_bias": {
        "type": "choice",
        "choices": ["liberal", "conservative", "neutral", "commercial_pro", "commercial_anti"],
        "instructions": "Identify the primary political or commercial bias.",
        "criteria": {
            "liberal": "Favors progressive policies and social change.",
            "conservative": "Favors traditional values and limited government.",
            "neutral": "Presents information objectively without a clear political or commercial leaning.",
            "commercial_pro": "Promotes a product, service, or company.",
            "commercial_anti": "Criticizes a product, service, or company."
        }
    },
    "factual_objectivity_rating": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the factual objectivity of the text from 1 (highly subjective) to 5 (highly objective)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Evaluating editorial bias with Jev...")
response = query_jev_model(state=news_article_state, questions=bias_guardrails_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    has_bias = answers.get('has_misleading_claims', {}).get('noul')
    bias_type = answers.get('political_commercial_bias', {}).get('choice')
    objectivity = answers.get('factual_objectivity_rating', {}).get('score')

    print(f"-> Editorial Guardrail Report:")
    print(f"   Misleading Content Prob:     {has_bias}")
    print(f"   Detected Stance/Bias:        {bias_type}")
    print(f"   Factual Objectivity Score:   {objectivity}")
else:
    print("[Error] Editorial guardrail request failed.")

[INFO] Evaluating editorial bias with Jev...
-> Editorial Guardrail Report:
   Misleading Content Prob:     0.96
   Detected Stance/Bias:        liberal
   Factual Objectivity Score:   0.04


In [49]:
response

{'answers': {'factual_objectivity_rating': {'confidence': 0.97,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.98, '1': 0.02, '2': 0, '3': 0, '4': 0},
   'score': 0.04,
   'type': 'score'},
  'has_misleading_claims': {'noul': 0.96, 'type': 'noul'},
  'political_commercial_bias': {'choice': 'liberal',
   'confidence': 0.98,
   'probabilities': {'commercial_anti': 0,
    'commercial_pro': 0,
    'conservative': 0.01,
    'liberal': 0.99,
    'neutral': 0},
   'type': 'choice'}},
 'id': 'task_qaFvrsoGfmPkp718rHgKVAPR5Q0HUgdK',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 566, 'output_tokens': 106}}

### Use Case 8: IT Security Incident Log Triage

This use case monitors server outputs and system events. It flags active cyber intrusions, classifies the vector of attack (such as brute force, SQL injection, or DDoS), and scores the overall operational severity for on-call engineers.

In [52]:
# --- Use Case 8: IT Security Incident Log Triage ---
# This use case monitors raw server messages to identify malicious threat activity,
# classify the exploit category, and rate severity to orchestrate emergency on-call pages.

security_log_state = (
    "2024-03-08 10:35:12 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:15 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:18 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "Attack detected from 203.0.113.44 on port 80 - potential SQL injection attempt. Alert sent."
)

it_security_questions = {
    "detect_intrusion_attempt": {
        "type": "noul",
        "instructions": "Does the log indicate an active intrusion attempt?",
        "criteria": {
            "true": "The log contains patterns indicative of unauthorized access, brute-force, or exploitation attempts.",
            "false": "The log shows routine activity or minor, non-malicious errors."
        }
    },
    "attack_type": {
        "type": "choice",
        "choices": ["DDoS", "brute_force", "SQL_injection", "malware_infection"],
        "instructions": "Classify the attack type.",
        "criteria": {
            "DDoS": "Distributed Denial of Service attack, overwhelming the system with traffic.",
            "brute_force": "Repeated, systematic attempts to guess credentials.",
            "SQL_injection": "Malicious code inserted into input fields to manipulate database queries.",
            "malware_infection": "System compromise by malicious software."
        }
    },
    "severity_level": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Prioritize the severity level for the on-call engineer from 1 (low) to 5 (critical)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Triaging security incident logs with Jev...")
response = query_jev_model(state=security_log_state, questions=it_security_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    intrusion_detected = answers.get('detect_intrusion_attempt', {}).get('noul')
    attack_category = answers.get('attack_type', {}).get('choice')
    severity = answers.get('severity_level', {}).get('score')

    print(f"-> Security Triage Report:")
    print(f"   Intrusion Probability:       {intrusion_detected}")
    print(f"   Identified Attack Category:  {attack_category}")
    print(f"   Assigned Severity (1-5):     {severity}")
else:
    print("[Error] IT security log triage failed.")

[INFO] Triaging security incident logs with Jev...
-> Security Triage Report:
   Intrusion Probability:       0.96
   Identified Attack Category:  SQL_injection
   Assigned Severity (1-5):     3.13


In [53]:
response

{'answers': {'attack_type': {'choice': 'SQL_injection',
   'confidence': 0.34,
   'probabilities': {'DDoS': 0,
    'SQL_injection': 0.5,
    'brute_force': 0.5,
    'malware_infection': 0},
   'type': 'choice'},
  'detect_intrusion_attempt': {'noul': 0.96, 'type': 'noul'},
  'severity_level': {'confidence': 0.7,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0.01, '2': 0.09, '3': 0.66, '4': 0.24},
   'score': 3.13,
   'type': 'score'}},
 'id': 'task_ePH88q0YJwLPdmqNyZ7cq6I9gAy4c6vA',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 679, 'output_tokens': 95}}

### Use Case 9: Medical Triage and Symptom Logging

This use case evaluates initial clinical intake descriptions to flag critical, life-threatening symptoms, recommend the most appropriate clinical specialty, and score the objective urgency of patient care from 1 to 5.

In [55]:
# --- Use Case 9: Medical Triage and Symptom Logging ---
# This use case evaluates patient symptoms to flag clinical emergencies,
# recommend medical departments, and prioritize care levels.

patient_notes_state = (
    "Patient presents with severe chest pain radiating to the left arm, "
    "shortness of breath, and dizziness. Onset 30 minutes ago. No prior history."
)

medical_triage_questions = {
    "flag_critical_symptoms": {
        "type": "noul",
        "instructions": "Does the patient exhibit critical life-threatening symptoms?",
        "criteria": {
            "true": "Symptoms indicate a severe medical emergency requiring immediate intervention (e.g., severe chest pain, difficulty breathing, sudden loss of consciousness).",
            "false": "Symptoms are non-life-threatening or mild."
        }
    },
    "medical_specialty": {
        "type": "choice",
        "choices": ["cardiology", "general_practice", "dermatology", "neurology", "emergency_medicine"],
        "instructions": "Recommend the appropriate medical specialty.",
        "criteria": {
            "cardiology": "Heart-related issues.",
            "general_practice": "Routine check-ups or common ailments.",
            "dermatology": "Skin conditions.",
            "neurology": "Brain and nervous system disorders.",
            "emergency_medicine": "Acute, life-threatening conditions."
        }
    },
    "urgency_of_care": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rank the urgency of care from 1 (routine) to 5 (immediate/critical)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Performing clinical intake triage with Jev...")
response = query_jev_model(state=patient_notes_state, questions=medical_triage_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    critical_flag = answers.get('flag_critical_symptoms', {}).get('noul')
    specialty = answers.get('medical_specialty', {}).get('choice')
    urgency = answers.get('urgency_of_care', {}).get('score')

    print(f"-> Medical Triage Assessment:")
    print(f"   Critical Symptom Probability: {critical_flag}")
    print(f"   Recommended Specialty:        {specialty}")
    print(f"   Assessed Urgency Rating:      {urgency}")
else:
    print("[Error] Medical triage request failed.")

[INFO] Performing clinical intake triage with Jev...
-> Medical Triage Assessment:
   Critical Symptom Probability: 0.97
   Recommended Specialty:        emergency_medicine
   Assessed Urgency Rating:      3.99


In [56]:
response

{'answers': {'flag_critical_symptoms': {'noul': 0.97, 'type': 'noul'},
  'medical_specialty': {'choice': 'emergency_medicine',
   'confidence': 0.99,
   'probabilities': {'cardiology': 0,
    'dermatology': 0,
    'emergency_medicine': 1,
    'general_practice': 0,
    'neurology': 0},
   'type': 'choice'},
  'urgency_of_care': {'confidence': 0.99,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0, '2': 0, '3': 0, '4': 1},
   'score': 3.99,
   'type': 'score'}},
 'id': 'task_jtgaJUL2GdUTcqKKsRFn1eXzPmK7toRh',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 546, 'output_tokens': 114}}

### Use Case 10: Contract and Legal Clause Risk Analysis

This use case accelerates legal compliance reviews by flagging non-standard liability clauses, identifying the exact legal domain affected (such as indemnity or intellectual property), and evaluating the company's financial risk exposure level.

In [58]:
# --- Use Case 10: Contract and Legal Clause Risk Analysis ---
# This use case highlights legal anomalies, categorizes target contract sections,
# and estimates company liability exposure.

legal_contract_state = (
    "This agreement includes a clause for unlimited liability for the vendor, "
    "which is non-standard. Payment terms are net 90 days. "
    "Intellectual property rights are assigned exclusively to the client."
)

legal_risk_questions = {
    "spot_non_standard_clauses": {
        "type": "noul",
        "instructions": "Does the contract contain non-standard or unusual liability clauses?",
        "criteria": {
            "true": "The contract includes terms that deviate significantly from industry norms regarding liability, indemnification, or warranty.",
            "false": "The contract's liability clauses are standard and acceptable."
        }
    },
    "legal_domain": {
        "type": "choice",
        "choices": ["indemnity", "payment_terms", "IP_rights", "termination_clauses"],
        "instructions": "Categorize the legal domain of the notable clause.",
        "criteria": {
            "indemnity": "Clauses related to protection against loss or damage.",
            "payment_terms": "Conditions for financial transactions and due dates.",
            "IP_rights": "Ownership and usage of intellectual property.",
            "termination_clauses": "Conditions under which the contract can be ended."
        }
    },
    "financial_exposure_risk": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Evaluate the overall financial exposure risk from 1 (low) to 5 (high)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Auditing legal contract with Jev...")
response = query_jev_model(state=legal_contract_state, questions=legal_risk_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    non_standard_flag = answers.get('spot_non_standard_clauses', {}).get('noul')
    domain = answers.get('legal_domain', {}).get('choice')
    exposure_score = answers.get('financial_exposure_risk', {}).get('score')

    print(f"-> Contract Compliance Report:")
    print(f"   Non-Standard Clauses Prob:   {non_standard_flag}")
    print(f"   Primary Affected Domain:     {domain}")
    print(f"   Financial Risk score:        {exposure_score}")
else:
    print("[Error] Contract auditing request failed.")

[INFO] Auditing legal contract with Jev...
-> Contract Compliance Report:
   Non-Standard Clauses Prob:   0.97
   Primary Affected Domain:     indemnity
   Financial Risk score:        3.65


In [59]:
response

{'answers': {'financial_exposure_risk': {'confidence': 0.7,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.01, '1': 0.04, '2': 0.04, '3': 0.11, '4': 0.8},
   'score': 3.65,
   'type': 'score'},
  'legal_domain': {'choice': 'indemnity',
   'confidence': 0.87,
   'probabilities': {'IP_rights': 0.08,
    'indemnity': 0.9,
    'payment_terms': 0.02,
    'termination_clauses': 0},
   'type': 'choice'},
  'spot_non_standard_clauses': {'noul': 0.97, 'type': 'noul'}},
 'id': 'task_xPpvfghEg017pIENVIi0hTxPCgE9GJZm',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 536, 'output_tokens': 96}}

# Use Case 11: Urgency test

In [61]:
# --- Use Case 11: Urgency Test ---
# This use case evaluates incoming user messages to flag immediate time-sensitive urgency.

urgency_message_state = (
    "Hi, I've been trying to connect my Stripe account for 3 days and it keeps failing. "
    "I'm losing sales. Please help ASAP."
)

urgency_questions = {
    "is_urgent": {
        "type": "noul",
        "instructions": "The message conveys urgency or time-sensitivity"
    }
}

print("[INFO] Evaluating urgency with Jev...")
response = query_jev_model(state=urgency_message_state, questions=urgency_questions)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    is_urgent = answers.get('is_urgent', {}).get('noul')
    print(f"-> Urgency Assessment:")
    print(f"   Urgency Score (0-1): {is_urgent}")
else:
    print("[Error] Urgency test request failed.")

[INFO] Evaluating urgency with Jev...
-> Urgency Assessment:
   Urgency Score (0-1): 0.99


In [62]:
response

{'answers': {'is_urgent': {'noul': 0.99, 'type': 'noul'}},
 'id': 'task_IXq54Z3eUBaIDN7bx6yhK83DA9K8m5rw',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 306, 'output_tokens': 23}}

# Use Case 12: Financial & Server Anomaly Detection

In [70]:
# --- Use Case 12: Financial & Server Anomaly Detection ---
# Simulates a mix of system metrics, metadata, and numerical arrays serialized as state,
# and runs parallel checks for threats, failure vectors, and threat severities.

import json

structured_metrics = {
    "account_id": "ACC-9921",
    "historical_baseline": {
        "avg_daily_transactions": 3,
        "avg_transaction_usd": 45.00,
        "max_allowed_hourly_limit": 500.00
    },
    "current_window_metrics": {
        "timestamp": "2026-09-29T17:50:00Z",
        "transactions_in_last_hour": 42,           # Massive spike vs baseline
        "total_usd_last_hour": 8450.00,            # Blew past max allowed limit
        "success_rate": 0.32,                      # Severe drop in success rate
        "error_codes_count": [
            {"code": 403, "count": 180},
            {"code": 500, "count": 12}
        ]
    },
    "device_ip_country": "Unknown/Proxy"
}

anomaly_questions = {
    "is_active_anomaly": {
        "type": "noul",
        "instructions": "Does the current window data indicate an extreme mathematical anomaly or potential security exploit compared to the historical baseline?"
    },
    "anomaly_type": {
        "type": "choice",
        "choices": ["Normal Behavior", "Credential Stuffing / Brute Force", "API Rate Exhaustion", "System Down / Technical Glitch"],
        "instructions": "Given the transaction count, crash counts, and drop in success rates, classify this anomaly pattern.",
        "criteria": {
            "Normal Behavior": "No deviations from standard performance guidelines or historic transactional ranges.",
            "Credential Stuffing / Brute Force": "High rates of authorization failures or localized 403 security errors.",
            "API Rate Exhaustion": "Spike in transaction counts far beyond historical limits from generic sources.",
            "System Down / Technical Glitch": "Severe drop in transaction success rates accompanied by high 500 series server codes."
        }
    },
    "threat_severity": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the operational threat level from 1 (Safe/Negligible) to 5 (Critical/Immediate Lockdown Required).",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Assessing systems anomaly indicators with Jev...")
# Querying Jev with the serialized metrics payload
response = query_jev_model(
    state=json.dumps(structured_metrics, indent=2),
    questions=anomaly_questions
)

if response and 'answers' in response:
    answers = response['answers']

    # Retrieve predictions safely
    is_anomaly = answers.get('is_active_anomaly', {}).get('noul')
    anomaly_pattern = answers.get('anomaly_type', {}).get('choice')
    threat_score = answers.get('threat_severity', {}).get('score')

    print(f"-> Operational Triage Report:")
    print(f"   Active Anomaly Indicator:  {is_anomaly}")
    print(f"   Classified Failure Vector: {anomaly_pattern}")
    print(f"   Risk Severity Rating:      {threat_score}")
else:
    print("[Error] Financial & server anomaly detection request failed.")

[INFO] Assessing systems anomaly indicators with Jev...
-> Operational Triage Report:
   Active Anomaly Indicator:  0.98
   Classified Failure Vector: Credential Stuffing / Brute Force
   Risk Severity Rating:      3.98


In [71]:
response

{'answers': {'anomaly_type': {'choice': 'Credential Stuffing / Brute Force',
   'confidence': 0.87,
   'probabilities': {'API Rate Exhaustion': 0.08,
    'Credential Stuffing / Brute Force': 0.91,
    'Normal Behavior': 0,
    'System Down / Technical Glitch': 0.01},
   'type': 'choice'},
  'is_active_anomaly': {'noul': 0.98, 'type': 'noul'},
  'threat_severity': {'confidence': 0.98,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0, '1': 0, '2': 0, '3': 0.02, '4': 0.98},
   'score': 3.98,
   'type': 'score'}},
 'id': 'task_98g4NEyLSlt5YCJ7WHjL27MSpLr0BJlJ',
 'model': 'jev-1.13-free',
 'usage': {'input_tokens': 717, 'output_tokens': 110}}

# Section 2: Laya Model

Here we show how the same classification and extraction patterns are simplified using the open-source Python library `laya`.

### About the Laya Routing Model

**Laya** is an open-source routing agent and classification model developed by Convai Innovations. It is specifically designed to handle structured text classification, categorical routing, and score prediction directly on device or via local deployment pipelines.

Key details and resources:
* **Hugging Face Repository**: You can find the model weights, configuration, and documentation on Hugging Face at [convaiinnovations/laya](https://huggingface.co/convaiinnovations/laya).
* **Local Execution**: Laya enables robust and fast 'System One' categorical classification and deterministic routing without relying on external network requests for every inference step.
* **Integration**: It integrates cleanly with the Python `laya` package, exposing a simple `Router` class to run batch questions.

In [66]:
!pip install laya -q

In [72]:
from laya import Router
from typing import Dict, Any, Optional

# Instantiate the global Router instance
laya_router = Router()

def query_laya_router(
    state: Any,
    questions: Dict[str, Any]
) -> Optional[Dict[str, Any]]:
    """
    Centralized production-grade wrapper to dispatch predictions using the Laya Router.

    Ensures uniform query invocation, localized error isolation, and
    safe output structures.
    """
    try:
        result = laya_router.predict(state, questions)
        return result
    except Exception as error:
        print(f"[Error] Laya router query failed: {error}")
        return None

# Simple test verification
test_state = {
    "subject": "Duplicate charge",
    "body": "We were billed twice for March. Please refund the duplicate today or we will cancel.",
}

test_questions = {
    "department": {
        "type": "choice",
        "instructions": "Which team should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, errors",
            "other": "everything else",
        },
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the customer explicitly threaten to cancel?",
    }
}

result = query_laya_router(test_state, test_questions)
if result and 'answers' in result:
    print(f"[Success] Verified Laya Route: {result['answers']['department']['choice']}")
    print(f"[Success] Verified Churn Indicator: {result['answers']['churn_risk']['noul']}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/router.py:926: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


[Success] Verified Laya Route: billing
[Success] Verified Churn Indicator: 0.8302


In [74]:
result

{'model': 'laya-rl-agent',
 'answers': {'department': {'type': 'choice',
   'choice': 'billing',
   'probabilities': {'billing': 0.9597, 'technical': 0.0223, 'other': 0.018},
   'confidence': 0.821,
   'answer_confidence': 0.9597,
   'action': {'act_probability': 1.0}},
  'churn_risk': {'type': 'noul',
   'noul': 0.8302,
   'confidence': 0.8302,
   'answer_confidence': 0.8302,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 132,
  'output_tokens': 0,
  'state_tokens': 30,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_questions': []},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/laya',
  'reason': 'English Latin text',
  'detection': {'script': 'latin',
   'script_profile': {'latin': 1.0},
   'language': 'en',
   'is_english': True,
   'language_undecided': False,
   'diacritic_rate': 0.0,
   'non_latin_fraction': 0.0,
   'mixed_segment': None},
  'workflow': None}}

## Laya Implementations of Use Cases

### Use Case 1: Customer Support Ticket Triage (Laya)
This use case demonstrates how to classify incoming customer support tickets. It assesses if an issue is an emergency, routes it to the appropriate department (e.g., Technical Support, Billing, Shipping), and estimates the customer's anger level, helping to prioritize and handle tickets efficiently.

In [75]:
# --- Use Case 1: Customer Support Ticket Triage (Laya) ---

customer_ticket_state_1 = (
    "My internet has been down for 3 days! I can't work from home. This is an emergency! "
    "I need someone to fix this NOW. "
)

support_triage_questions_1 = {
    "is_emergency": {
        "type": "noul",
        "instructions": "Is the customer issue an active emergency?",
        "criteria": {
            "true": "The issue requires immediate, critical attention due to severe impact or danger.",
            "false": "The issue is standard and does not pose an immediate threat or severe disruption."
        }
    },
    "department_routing": {
        "type": "choice",
        "choices": ["Billing", "Technical Support", "Shipping"],
        "instructions": "Which department should handle this ticket?",
        "criteria": {
            "Billing": "Deals with invoices, payments, and account charges.",
            "Technical Support": "Handles product malfunctions, software issues, or technical guidance.",
            "Shipping": "Manages delivery, tracking, and logistics."
        }
    },
    "anger_level": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the customer's anger level from 1 (calm) to 5 (enraged)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

print("[INFO] Querying Laya Router...")
result_1 = query_laya_router(customer_ticket_state_1, support_triage_questions_1)

if result_1 and 'answers' in result_1:
    answers = result_1['answers']
    is_emergency_1 = answers.get('is_emergency', {}).get('noul')
    department_routing_1 = answers.get('department_routing', {}).get('choice')
    anger_level_1 = answers.get('anger_level', {}).get('score')

    print(f"Is Emergency: {is_emergency_1}")
    print(f"Department Routing: {department_routing_1}")
    print(f"Anger Level: {anger_level_1}")
else:
    print("[Error] Failed to resolve predictions with Laya.")

[INFO] Querying Laya Router...
Is Emergency: 0.8716
Department Routing: Technical Support
Anger Level: 0.4388


In [76]:
result_1

{'model': 'laya-rl-agent',
 'answers': {'is_emergency': {'type': 'noul',
   'noul': 0.8716,
   'confidence': 0.8716,
   'answer_confidence': 0.8716,
   'action': {'act_probability': 1.0}},
  'department_routing': {'type': 'choice',
   'choice': 'Technical Support',
   'probabilities': {'Billing': 0.1166,
    'Technical Support': 0.6927,
    'Shipping': 0.1907},
   'confidence': 0.2528,
   'answer_confidence': 0.6927,
   'action': {'act_probability': 1.0}},
  'anger_level': {'type': 'score',
   'score': 0.4388,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.7158,
    '1': 0.1651,
    '2': 0.0904,
    '3': 0.0218,
    '4': 0.0068},
   'confidence': 0.4584,
   'answer_confidence': 0.7158,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 251,
  'output_tokens': 0,
  'state_tokens': 30,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_questions': []},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations

### Use Case 2: Content Moderation and Brand Safety (Laya)
This use case focuses on moderating user-generated content for brand safety. It detects hate speech, categorizes the type of violation (e.g., harassment, profanity), and rates the overall toxicity level of the content, which is crucial for maintaining a safe online environment.

In [78]:
# --- Use Case 2: Content Moderation and Brand Safety (Laya) ---

user_comment_state_2 = "This website is terrible and anyone who uses it is an idiot. You should all go to hell!"

content_moderation_questions_2 = {
    "flag_hate_speech": {
        "type": "noul",
        "instructions": "Does this comment contain explicit hate speech?",
        "criteria": {
            "true": "The comment contains derogatory, discriminatory, or hateful language targeting a specific group or individual.",
            "false": "The comment does not contain explicit hate speech."
        }
    },
    "violation_type": {
        "type": "choice",
        "choices": ["spam", "harassment", "profanity", "hate_speech"],
        "instructions": "Categorize the primary violation type.",
        "criteria": {
            "spam": "Unsolicited or repetitive content.",
            "harassment": "Targeted abusive behavior.",
            "profanity": "Offensive language.",
            "hate_speech": "Expressions of hatred, discrimination, or violence against a protected group."
        }
    },
    "toxicity_level": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Rate the toxicity level from 1 (mild) to 10 (extreme)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

print("[INFO] Querying Laya Router for Content Moderation...")
result_2 = query_laya_router(user_comment_state_2, content_moderation_questions_2)

if result_2 and 'answers' in result_2:
    answers = result_2['answers']
    flag_hate_speech_2 = answers.get('flag_hate_speech', {}).get('noul')
    violation_type_2 = answers.get('violation_type', {}).get('choice')
    toxicity_level_2 = answers.get('toxicity_level', {}).get('score')

    print(f"Flag Hate Speech: {flag_hate_speech_2}")
    print(f"Violation Category: {violation_type_2}")
    print(f"Toxicity Score: {toxicity_level_2}")
else:
    print("[Error] Failed to resolve predictions with Laya.")

[INFO] Querying Laya Router for Content Moderation...
Flag Hate Speech: 0.2183
Violation Category: harassment
Toxicity Score: 0.984


In [79]:
result_2

{'model': 'laya-rl-agent',
 'answers': {'flag_hate_speech': {'type': 'noul',
   'noul': 0.2183,
   'confidence': 0.7817,
   'answer_confidence': 0.7817,
   'action': {'act_probability': 1.0}},
  'violation_type': {'type': 'choice',
   'choice': 'harassment',
   'probabilities': {'spam': 0.1762,
    'harassment': 0.5622,
    'profanity': 0.208,
    'hate_speech': 0.0536},
   'confidence': 0.197,
   'answer_confidence': 0.5622,
   'action': {'act_probability': 1.0}},
  'toxicity_level': {'type': 'score',
   'score': 0.984,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0.8287,
    '1': 0.0399,
    '2': 0.0107,
    '3': 0.0086,
    '4': 0.0051,
    '5': 0.0052,
    '6': 0.0061,
    '7': 0.0099,
    '8': 0.0272,
    '9': 0.0585},
   'confidence': 0.666,
   'answer_confidence': 0.8287,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 246,
  'o

### Use Case 3: Resume Screening and Candidate Pre-Qualification (Laya)
This use case helps in quickly pre-qualifying job applicants based on their resumes. It checks if mandatory experience requirements are met, determines the candidate's seniority tier (Junior, Mid, Senior, Lead), and rates their overall skill alignment for a specific role.

In [80]:
from laya import Router

router = Router()

resume_text_state_3 = (
    "John Doe. 10 years experience in Machine Learning. "
    "Led 3 major AI projects. PhD in Computer Science. "
    "Skills: Python, TensorFlow, PyTorch, AWS, Azure."
)

resume_screening_questions_3 = {
    "mandatory_experience_met": {
        "type": "noul",
        "instructions": "Does the applicant have at least 5 years of experience in Machine Learning?",
        "criteria": {
            "true": "Applicant explicitly states or implies 5 or more years of relevant experience.",
            "false": "Applicant states less than 5 years or does not provide enough information."
        }
    },
    "seniority_tier": {
        "type": "choice",
        "choices": ["Junior", "Mid", "Senior", "Lead"],
        "instructions": "Categorize the candidate's seniority tier.",
        "criteria": {
            "Junior": "Less than 3 years of experience.",
            "Mid": "3-5 years of experience.",
            "Senior": "5+ years of experience with leadership potential.",
            "Lead": "Extensive experience, proven leadership, and project management skills."
        }
    },
    "skill_alignment": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Rate the overall skill alignment with a typical Senior Machine Learning Engineer role (1-low, 10-high)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

result_3 = router.predict(resume_text_state_3, resume_screening_questions_3)

mandatory_experience_met_3 = result_3['answers']['mandatory_experience_met']['noul']
seniority_tier_3 = result_3['answers']['seniority_tier']['choice']
skill_alignment_3 = result_3['answers']['skill_alignment']['score']

print(f"Mandatory Experience Met: {mandatory_experience_met_3}")
print(f"Seniority Tier: {seniority_tier_3}")
print(f"Skill Alignment: {skill_alignment_3}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/router.py:926: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


Mandatory Experience Met: 0.7308
Seniority Tier: Lead
Skill Alignment: 5.6602


In [81]:
result_3

{'model': 'laya-rl-agent',
 'answers': {'mandatory_experience_met': {'type': 'noul',
   'noul': 0.7308,
   'confidence': 0.7308,
   'answer_confidence': 0.7308,
   'action': {'act_probability': 1.0}},
  'seniority_tier': {'type': 'choice',
   'choice': 'Lead',
   'probabilities': {'Junior': 0.0431,
    'Mid': 0.1595,
    'Senior': 0.3498,
    'Lead': 0.4477},
   'confidence': 0.1665,
   'answer_confidence': 0.4477,
   'action': {'act_probability': 1.0}},
  'skill_alignment': {'type': 'score',
   'score': 5.6602,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0.0442,
    '1': 0.0695,
    '2': 0.0885,
    '3': 0.0988,
    '4': 0.045,
    '5': 0.057,
    '6': 0.0854,
    '7': 0.1225,
    '8': 0.2193,
    '9': 0.1698},
   'confidence': 0.0574,
   'answer_confidence': 0.2193,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 308,
  'output_toke

### Use Case 4: E-Commerce Fake Review Detection (Laya)
This use case aims to identify potentially fraudulent or incentivized product reviews on e-commerce platforms. It flags suspicious reviews, classifies the primary focus of the review (e.g., quality, price), and estimates its overall authenticity rating.

In [82]:
from laya import Router

router = Router()

product_review_state_4 = (
    "This product is absolutely amazing! I got it for free "
    "as part of a promotion. Best purchase ever!"
)

fake_review_questions_4 = {
    "flag_fraudulent_review": {
        "type": "noul",
        "instructions": "Is this review potentially fraudulent or incentivized?",
        "criteria": {
            "true": "The review mentions receiving the product for free, in exchange for a review, or shows other signs of unnatural positivity or repetition.",
            "false": "The review appears genuine and organic."
        }
    },
    "review_focus": {
        "type": "choice",
        "choices": ["quality", "price", "shipping", "customer_service"],
        "instructions": "Classify the primary focus of the review.",
        "criteria": {
            "quality": "Focuses on the product's durability, performance, or materials.",
            "price": "Comments on the cost, value for money, or deals.",
            "shipping": "Mentions delivery speed, packaging, or logistics.",
            "customer_service": "Refers to interactions with support or sales."
        }
    },
    "authenticity_rating": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Estimate the authenticity rating from 1 (very fake) to 5 (very authentic)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_4 = router.predict(product_review_state_4, fake_review_questions_4)

flag_fraudulent_review_4 = result_4['answers']['flag_fraudulent_review']['noul']
review_focus_4 = result_4['answers']['review_focus']['choice']
authenticity_rating_4 = result_4['answers']['authenticity_rating']['score']

print(f"Flag Fraudulent Review: {flag_fraudulent_review_4}")
print(f"Review Focus: {review_focus_4}")
print(f"Authenticity Rating: {authenticity_rating_4}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Flag Fraudulent Review: 0.181
Review Focus: quality
Authenticity Rating: 2.0093


In [83]:
result_4

{'model': 'laya-rl-agent',
 'answers': {'flag_fraudulent_review': {'type': 'noul',
   'noul': 0.181,
   'confidence': 0.819,
   'answer_confidence': 0.819,
   'action': {'act_probability': 1.0}},
  'review_focus': {'type': 'choice',
   'choice': 'quality',
   'probabilities': {'quality': 0.441,
    'price': 0.1291,
    'shipping': 0.2362,
    'customer_service': 0.1937},
   'confidence': 0.0737,
   'answer_confidence': 0.441,
   'action': {'act_probability': 1.0}},
  'authenticity_rating': {'type': 'score',
   'score': 2.0093,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.2655,
    '1': 0.1507,
    '2': 0.1432,
    '3': 0.1903,
    '4': 0.2503},
   'confidence': 0.0195,
   'answer_confidence': 0.2655,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 242,
  'output_tokens': 0,
  'state_tokens': 21,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_questions': []},
 'routing': {'model': 'english',
  'repo': '

### Use Case 5: Sentiment Analysis of Social Media Posts (Laya)
This use case analyzes the sentiment expressed in social media posts or comments. It determines if the sentiment is negative, categorizes the overall sentiment (positive, negative, neutral), and rates the intensity of that sentiment.

In [84]:
from laya import Router

router = Router()

social_media_post_state_5 = (
    "Just had the worst experience with my new phone. The battery dies so fast, "
    "and the camera is terrible. What a waste of money! #disappointed #techfail"
)

sentiment_analysis_questions_5 = {
    "is_negative_sentiment": {
        "type": "noul",
        "instructions": "Does the social media post express negative sentiment?",
        "criteria": {
            "true": "The post contains words, phrases, or overall tone indicating dissatisfaction, anger, or sadness.",
            "false": "The post is neutral or positive in sentiment."
        }
    },
    "sentiment_category": {
        "type": "choice",
        "choices": ["positive", "negative", "neutral"],
        "instructions": "Categorize the overall sentiment of the post.",
        "criteria": {
            "positive": "Expresses approval, happiness, or satisfaction.",
            "negative": "Expresses disapproval, sadness, or frustration.",
            "neutral": "No strong positive or negative emotion is conveyed."
        }
    },
    "sentiment_intensity": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the intensity of the sentiment from 1 (mild) to 5 (extreme)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_5 = router.predict(social_media_post_state_5, sentiment_analysis_questions_5)

is_negative_sentiment_5 = result_5['answers']['is_negative_sentiment']['noul']
sentiment_category_5 = result_5['answers']['sentiment_category']['choice']
sentiment_intensity_5 = result_5['answers']['sentiment_intensity']['score']

print(f"Is Negative Sentiment: {is_negative_sentiment_5}")
print(f"Sentiment Category: {sentiment_category_5}")
print(f"Sentiment Intensity: {sentiment_intensity_5}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Is Negative Sentiment: 0.8947
Sentiment Category: negative
Sentiment Intensity: 0.2956


In [85]:
result_5

{'model': 'laya-rl-agent',
 'answers': {'is_negative_sentiment': {'type': 'noul',
   'noul': 0.8947,
   'confidence': 0.8947,
   'answer_confidence': 0.8947,
   'action': {'act_probability': 1.0}},
  'sentiment_category': {'type': 'choice',
   'choice': 'negative',
   'probabilities': {'positive': 0.0122,
    'negative': 0.9725,
    'neutral': 0.0153},
   'confidence': 0.8683,
   'answer_confidence': 0.9725,
   'action': {'act_probability': 1.0}},
  'sentiment_intensity': {'type': 'score',
   'score': 0.2956,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.7333,
    '1': 0.2459,
    '2': 0.0153,
    '3': 0.003,
    '4': 0.0025},
   'confidence': 0.5844,
   'answer_confidence': 0.7333,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 258,
  'output_tokens': 0,
  'state_tokens': 34,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_questions': []},
 'routing': {'model': 'english',
  'repo': 'convaiinnovations/l

### Use Case 6: Sales Lead Qualification and Intent Scoring (Laya)
This use case evaluates sales leads to determine their qualification and purchase intent. It identifies if a lead has an immediate budget, classifies their stage in the buyer's journey, and predicts their conversion potential.

In [86]:
from laya import Router

router = Router()

sales_lead_state_6 = (
    "Interested in your enterprise solution. Our budget is $100k, "
    "and we're looking to implement within the next 3 months. "
    "Currently evaluating vendors."
)

lead_qualification_questions_6 = {
    "has_immediate_budget": {
        "type": "noul",
        "instructions": "Does the lead have an immediate budget for the solution?",
        "criteria": {
            "true": "The lead explicitly states a budget or indicates readiness for financial commitment.",
            "false": "The lead does not mention a budget or indicates they are still early in the financial planning phase."
        }
    },
    "buyer_journey_stage": {
        "type": "choice",
        "choices": ["researching", "comparing", "ready_to_buy"],
        "instructions": "Classify the lead's buyer journey stage.",
        "criteria": {
            "researching": "Gathering information and understanding options.",
            "comparing": "Evaluating different solutions and vendors.",
            "ready_to_buy": "Prepared to make a purchase decision and looking for final quotes."
        }
    },
    "conversion_potential": {
        "type": "score",
        "min": 1,
        "max": 10,
        "instructions": "Grade the lead's conversion potential from 1 (low) to 10 (high)",
        "criteria": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
    }
}

result_6 = router.predict(sales_lead_state_6, lead_qualification_questions_6)

has_immediate_budget_6 = result_6['answers']['has_immediate_budget']['noul']
buyer_journey_stage_6 = result_6['answers']['buyer_journey_stage']['choice']
conversion_potential_6 = result_6['answers']['conversion_potential']['score']

print(f"Has Immediate Budget: {has_immediate_budget_6}")
print(f"Buyer Journey Stage: {buyer_journey_stage_6}")
print(f"Conversion Potential: {conversion_potential_6}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Has Immediate Budget: 0.7159
Buyer Journey Stage: comparing
Conversion Potential: 6.0671


In [87]:
result_6

{'model': 'laya-rl-agent',
 'answers': {'has_immediate_budget': {'type': 'noul',
   'noul': 0.7159,
   'confidence': 0.7159,
   'answer_confidence': 0.7159,
   'action': {'act_probability': 1.0}},
  'buyer_journey_stage': {'type': 'choice',
   'choice': 'comparing',
   'probabilities': {'researching': 0.3063,
    'comparing': 0.3635,
    'ready_to_buy': 0.3302},
   'confidence': 0.0022,
   'answer_confidence': 0.3635,
   'action': {'act_probability': 1.0}},
  'conversion_potential': {'type': 'score',
   'score': 6.0671,
   'legend': {'0': '1',
    '1': '2',
    '2': '3',
    '3': '4',
    '4': '5',
    '5': '6',
    '6': '7',
    '7': '8',
    '8': '9',
    '9': '10'},
   'probabilities': {'0': 0.0216,
    '1': 0.0428,
    '2': 0.064,
    '3': 0.0913,
    '4': 0.0583,
    '5': 0.0734,
    '6': 0.1173,
    '7': 0.1307,
    '8': 0.2025,
    '9': 0.1983},
   'confidence': 0.0745,
   'answer_confidence': 0.2025,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 276,
  'ou

### Use Case 7: News and Article Bias Guardrails (Laya)
This use case applies guardrails to news articles and similar content to detect potential biases. It flags misleading claims or unsubstantiated accusations, identifies political or commercial bias, and rates the factual objectivity of the text.

In [88]:
from laya import Router

router = Router()

news_article_state_7 = (
    "Breaking news: The government's new policy, widely condemned by experts, "
    "is clearly an attempt to suppress dissent and control information. "
    "This authoritarian move will undoubtedly lead to widespread protests."
)

bias_guardrails_questions_7 = {
    "has_misleading_claims": {
        "type": "noul",
        "instructions": "Does the text contain misleading claims or unsubstantiated accusations?",
        "criteria": {
            "true": "The text presents opinions as facts, uses loaded language, or makes claims without evidence.",
            "false": "The text is factual, balanced, and avoids deceptive language."
        }
    },
    "political_commercial_bias": {
        "type": "choice",
        "choices": ["liberal", "conservative", "neutral", "commercial_pro", "commercial_anti"],
        "instructions": "Identify the primary political or commercial bias.",
        "criteria": {
            "liberal": "Favors progressive policies and social change.",
            "conservative": "Favors traditional values and limited government.",
            "neutral": "Presents information objectively without a clear political or commercial leaning.",
            "commercial_pro": "Promotes a product, service, or company.",
            "commercial_anti": "Criticizes a product, service, or company."
        }
    },
    "factual_objectivity_rating": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rate the factual objectivity of the text from 1 (highly subjective/biased) to 5 (highly objective/unbiased)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_7 = router.predict(news_article_state_7, bias_guardrails_questions_7)

has_misleading_claims_7 = result_7['answers']['has_misleading_claims']['noul']
political_commercial_bias_7 = result_7['answers']['political_commercial_bias']['choice']
factual_objectivity_rating_7 = result_7['answers']['factual_objectivity_rating']['score']

print(f"Has Misleading Claims: {has_misleading_claims_7}")
print(f"Political/Commercial Bias: {political_commercial_bias_7}")
print(f"Factual Objectivity Rating: {factual_objectivity_rating_7}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Has Misleading Claims: 0.2304
Political/Commercial Bias: neutral
Factual Objectivity Rating: 0.5718


In [89]:
result_7

{'model': 'laya-rl-agent',
 'answers': {'has_misleading_claims': {'type': 'noul',
   'noul': 0.2304,
   'confidence': 0.7696,
   'answer_confidence': 0.7696,
   'action': {'act_probability': 1.0}},
  'political_commercial_bias': {'type': 'choice',
   'choice': 'neutral',
   'probabilities': {'liberal': 0.1437,
    'conservative': 0.2603,
    'neutral': 0.3371,
    'commercial_pro': 0.0601,
    'commercial_anti': 0.1988},
   'confidence': 0.0768,
   'answer_confidence': 0.3371,
   'action': {'act_probability': 1.0}},
  'factual_objectivity_rating': {'type': 'score',
   'score': 0.5718,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.5805,
    '1': 0.3107,
    '2': 0.0753,
    '3': 0.0235,
    '4': 0.01},
   'confidence': 0.3738,
   'answer_confidence': 0.5805,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 308,
  'output_tokens': 0,
  'state_tokens': 36,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_ques

### Use Case 8: IT Security Incident Log Triage (Laya)
This use case assists in triaging IT security incidents from log data. It detects active intrusion attempts, classifies the type of attack (e.g., brute_force, SQL_injection), and assigns a severity level for immediate action by on-call engineers.

In [90]:
from laya import Router

router = Router()

security_log_state_8 = (
    "2024-03-08 10:35:12 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:15 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "2024-03-08 10:35:18 - IP: 192.168.1.10 - User 'admin' failed login 5 times. "
    "Attack detected from 203.0.113.44 on port 80 - potential SQL injection attempt. Alert sent."
)

it_security_questions_8 = {
    "detect_intrusion_attempt": {
        "type": "noul",
        "instructions": "Does the log indicate an active intrusion attempt?",
        "criteria": {
            "true": "The log contains patterns indicative of unauthorized access, brute-force, or exploitation attempts.",
            "false": "The log shows routine activity or minor, non-malicious errors."
        }
    },
    "attack_type": {
        "type": "choice",
        "choices": ["DDoS", "brute_force", "SQL_injection", "malware_infection"],
        "instructions": "Classify the attack type.",
        "criteria": {
            "DDoS": "Distributed Denial of Service attack, overwhelming the system with traffic.",
            "brute_force": "Repeated, systematic attempts to guess credentials.",
            "SQL_injection": "Malicious code inserted into input fields to manipulate database queries.",
            "malware_infection": "System compromise by malicious software."
        }
    },
    "severity_level": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Prioritize the severity level for the on-call engineer from 1 (low) to 5 (critical)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_8 = router.predict(security_log_state_8, it_security_questions_8)

detect_intrusion_attempt_8 = result_8['answers']['detect_intrusion_attempt']['noul']
attack_type_8 = result_8['answers']['attack_type']['choice']
severity_level_8 = result_8['answers']['severity_level']['score']

print(f"Detect Intrusion Attempt: {detect_intrusion_attempt_8}")
print(f"Attack Type: {attack_type_8}")
print(f"Severity Level: {severity_level_8}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Detect Intrusion Attempt: 0.899
Attack Type: DDoS
Severity Level: 1.517


In [91]:
result_8

{'model': 'laya-rl-agent',
 'answers': {'detect_intrusion_attempt': {'type': 'noul',
   'noul': 0.899,
   'confidence': 0.899,
   'answer_confidence': 0.899,
   'action': {'act_probability': 1.0}},
  'attack_type': {'type': 'choice',
   'choice': 'DDoS',
   'probabilities': {'DDoS': 0.5373,
    'brute_force': 0.1163,
    'SQL_injection': 0.257,
    'malware_infection': 0.0893},
   'confidence': 0.1712,
   'answer_confidence': 0.5373,
   'action': {'act_probability': 1.0}},
  'severity_level': {'type': 'score',
   'score': 1.517,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.2573,
    '1': 0.4084,
    '2': 0.0912,
    '3': 0.0462,
    '4': 0.1969},
   'confidence': 0.1329,
   'answer_confidence': 0.4084,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 526,
  'output_tokens': 0,
  'state_tokens': 115,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_questions': []},
 'routing': {'model': 'english',
  'repo'

### Use Case 9: Medical Triage and Symptom Logging (Laya)
This use case helps in the initial triage of medical patients based on their intake notes. It flags critical life-threatening symptoms, recommends the appropriate medical specialty for referral, and ranks the urgency of care required.

In [92]:
from laya import Router

router = Router()

patient_notes_state_9 = (
    "Patient presents with severe chest pain radiating to the left arm, "
    "shortness of breath, and dizziness. Onset 30 minutes ago. No prior history."
)

medical_triage_questions_9 = {
    "flag_critical_symptoms": {
        "type": "noul",
        "instructions": "Does the patient exhibit critical life-threatening symptoms?",
        "criteria": {
            "true": "Symptoms indicate a severe medical emergency requiring immediate intervention (e.g., severe chest pain, difficulty breathing, sudden loss of consciousness).",
            "false": "Symptoms are non-life-threatening or mild."
        }
    },
    "medical_specialty": {
        "type": "choice",
        "choices": ["cardiology", "general_practice", "dermatology", "neurology", "emergency_medicine"],
        "instructions": "Recommend the appropriate medical specialty.",
        "criteria": {
            "cardiology": "Heart-related issues.",
            "general_practice": "Routine check-ups or common ailments.",
            "dermatology": "Skin conditions.",
            "neurology": "Brain and nervous system disorders.",
            "emergency_medicine": "Acute, life-threatening conditions."
        }
    },
    "urgency_of_care": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Rank the urgency of care from 1 (routine) to 5 (immediate/critical)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_9 = router.predict(patient_notes_state_9, medical_triage_questions_9)

flag_critical_symptoms_9 = result_9['answers']['flag_critical_symptoms']['noul']
medical_specialty_9 = result_9['answers']['medical_specialty']['choice']
urgency_of_care_9 = result_9['answers']['urgency_of_care']['score']

print(f"Flag Critical Symptoms: {flag_critical_symptoms_9}")
print(f"Medical Specialty: {medical_specialty_9}")
print(f"Urgency of Care: {urgency_of_care_9}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Flag Critical Symptoms: 0.7856
Medical Specialty: emergency_medicine
Urgency of Care: 0.9712


In [93]:
result_9

{'model': 'laya-rl-agent',
 'answers': {'flag_critical_symptoms': {'type': 'noul',
   'noul': 0.7856,
   'confidence': 0.7856,
   'answer_confidence': 0.7856,
   'action': {'act_probability': 1.0}},
  'medical_specialty': {'type': 'choice',
   'choice': 'emergency_medicine',
   'probabilities': {'cardiology': 0.2288,
    'general_practice': 0.0849,
    'dermatology': 0.0259,
    'neurology': 0.0422,
    'emergency_medicine': 0.6183},
   'confidence': 0.3338,
   'answer_confidence': 0.6183,
   'action': {'act_probability': 1.0}},
  'urgency_of_care': {'type': 'score',
   'score': 0.9712,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.4212,
    '1': 0.3957,
    '2': 0.0685,
    '3': 0.0197,
    '4': 0.0949},
   'confidence': 0.2448,
   'answer_confidence': 0.4212,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 276,
  'output_tokens': 0,
  'state_tokens': 32,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_

### Use Case 10: Contract and Legal Clause Risk Analysis (Laya)
This use case analyzes legal contracts to identify potential risks. It spots non-standard or unusual liability clauses, categorizes the legal domain of any notable clauses, and evaluates the overall financial exposure risk associated with the contract.

In [94]:
from laya import Router

router = Router()

legal_contract_state_10 = (
    "This agreement includes a clause for unlimited liability for the vendor, "
    "which is non-standard. Payment terms are net 90 days. "
    "Intellectual property rights are assigned exclusively to the client."
)

legal_risk_questions_10 = {
    "spot_non_standard_clauses": {
        "type": "noul",
        "instructions": "Does the contract contain non-standard or unusual liability clauses?",
        "criteria": {
            "true": "The contract includes terms that deviate significantly from industry norms regarding liability, indemnification, or warranty.",
            "false": "The contract's liability clauses are standard and acceptable."
        }
    },
    "legal_domain": {
        "type": "choice",
        "choices": ["indemnity", "payment_terms", "IP_rights", "termination_clauses"],
        "instructions": "Categorize the legal domain of the notable clause.",
        "criteria": {
            "indemnity": "Clauses related to protection against loss or damage.",
            "payment_terms": "Conditions for financial transactions and due dates.",
            "IP_rights": "Ownership and usage of intellectual property.",
            "termination_clauses": "Conditions under which the contract can be ended."
        }
    },
    "financial_exposure_risk": {
        "type": "score",
        "min": 1,
        "max": 5,
        "instructions": "Evaluate the overall financial exposure risk from 1 (low) to 5 (high)",
        "criteria": ["1", "2", "3", "4", "5"]
    }
}

result_10 = router.predict(legal_contract_state_10, legal_risk_questions_10)

spot_non_standard_clauses_10 = result_10['answers']['spot_non_standard_clauses']['noul']
legal_domain_10 = result_10['answers']['legal_domain']['choice']
financial_exposure_risk_10 = result_10['answers']['financial_exposure_risk']['score']

print(f"Spot Non-Standard Clauses: {spot_non_standard_clauses_10}")
print(f"Legal Domain: {legal_domain_10}")
print(f"Financial Exposure Risk: {financial_exposure_risk_10}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Spot Non-Standard Clauses: 0.8679
Legal Domain: payment_terms
Financial Exposure Risk: 1.0557


In [95]:
result_10

{'model': 'laya-rl-agent',
 'answers': {'spot_non_standard_clauses': {'type': 'noul',
   'noul': 0.8679,
   'confidence': 0.8679,
   'answer_confidence': 0.8679,
   'action': {'act_probability': 1.0}},
  'legal_domain': {'type': 'choice',
   'choice': 'payment_terms',
   'probabilities': {'indemnity': 0.0296,
    'payment_terms': 0.8444,
    'IP_rights': 0.0904,
    'termination_clauses': 0.0356},
   'confidence': 0.5795,
   'answer_confidence': 0.8444,
   'action': {'act_probability': 1.0}},
  'financial_exposure_risk': {'type': 'score',
   'score': 1.0557,
   'legend': {'0': '1', '1': '2', '2': '3', '3': '4', '4': '5'},
   'probabilities': {'0': 0.398,
    '1': 0.3168,
    '2': 0.1725,
    '3': 0.0568,
    '4': 0.0559},
   'confidence': 0.1562,
   'answer_confidence': 0.398,
   'action': {'act_probability': 1.0}}},
 'usage': {'input_tokens': 287,
  'output_tokens': 0,
  'state_tokens': 36,
  'state_tokens_dropped': 0,
  'truncated': False,
  'truncated_questions': []},
 'routing': {'

## Conclusion: Jev vs. Laya

In this notebook, we explored and implemented production-grade classification, risk scoring, and routing workflows using two distinct System One model families:

1. **Jev (by Typesafe)**: A commercial, state-of-the-art hosted API. Jev delivers highly calibrated and precise distributions with complex, rich configurations (such as score mappings and choice triages) processed on powerful cloud infrastructure via fast HTTP integration.
2. **Laya (by Convai Innovations)**: A completely **free and open-source** local routing agent. Laya allows you to run local inference directly on-device without external API limits, making it perfect for cost-effective scaling, private deployments, and fast pipelines.

### Summary
Both frameworks bypass verbose System Two reasoning to provide instantaneous, structured categorical decisions, showing how developers can choose between premium cloud calibration (**Jev**) or open-source local sovereignty (**Laya**) depending on production requirements.


---